In [4]:
import pandas as pd
import os
import sys
import tomllib 

sys.path.append(os.path.abspath("../.secrets"))
from db_config import get_connection

print("Se conecto correctamente a la Base de Datos")

Se conecto correctamente a la Base de Datos


In [5]:
with open("../.secrets/secrets.toml", "rb") as f:
    config = tomllib.load(f)

ruta_archivos = config["paths"]["bronze_path"]

print("Ruta configurada:", ruta_archivos)

Ruta configurada: assets/docs


In [6]:
conn = get_connection()
cursor = conn.cursor()

print("Conexión exitosa")

Conexión exitosa


In [7]:
archivos = [f for f in os.listdir(f"../{ruta_archivos}") if f.endswith(".csv")]

print("\nArchivos disponibles:")
for i, archivo in enumerate(archivos):
    print(f"{i} - {archivo}")

entrada = input("\nSelecciona el número del archivo que deseas cargar: ")

# Validar que sea número
if not entrada.isdigit():
    print("Debes ingresar un número válido")
    exit()

indice = int(entrada)

# Validar rango
if indice < 0 or indice >= len(archivos):
    print("Número inválido")
    exit()

archivo_seleccionado = archivos[indice]

print(f"\nCargando archivo: {archivo_seleccionado}")


Archivos disponibles:
0 - clientes.csv
1 - productos.csv
2 - ventas.csv

Cargando archivo: clientes.csv


In [8]:
from pathlib import Path
from io import StringIO
from datetime import datetime

# -----------------------------
# VALIDACIONES
# -----------------------------
if "archivo_seleccionado" not in globals():
    raise Exception("Primero selecciona el archivo.")

tabla_destino = archivo_seleccionado.replace(".csv", "_raw")
print(f"Tabla destino: bronze.{tabla_destino}")

# -----------------------------
# LEER CSV
# -----------------------------
ruta_completa = Path(f"../{ruta_archivos}") / archivo_seleccionado
df = pd.read_csv(ruta_completa)

# Agregar metadata
df["fuente_archivo"] = archivo_seleccionado
df["fecha_carga"] = datetime.now()

# -----------------------------
# OBTENER COLUMNAS REALES DE LA TABLA
# -----------------------------
cursor.execute(f"""
SELECT column_name
FROM information_schema.columns
WHERE table_schema = 'bronze'
AND table_name = '{tabla_destino}'
ORDER BY ordinal_position;
""")

columnas_tabla = [row[0] for row in cursor.fetchall()]

print("Columnas en Supabase:", columnas_tabla)

# -----------------------------
# AJUSTAR DATAFRAME A LA TABLA
# -----------------------------

# Agregar columnas faltantes
for col in columnas_tabla:
    if col not in df.columns:
        df[col] = None

# Mantener solo las columnas que existen en la tabla
df = df[columnas_tabla]

# -----------------------------
# COPY
# -----------------------------
buffer = StringIO()
df.to_csv(buffer, index=False, header=False)
buffer.seek(0)

copy_sql = f"""
COPY bronze.{tabla_destino}
FROM STDIN WITH CSV
"""

try:
    cursor.copy_expert(copy_sql, buffer)
    conn.commit()
    print("Carga completada correctamente")

except Exception as e:
    conn.rollback()
    print("Error durante la carga:", e)

finally:
    cursor.close()
    conn.close()

Tabla destino: bronze.clientes_raw
Columnas en Supabase: ['id_cliente', 'nombre_cliente', 'genero', 'edad', 'ciudad', 'pais', 'segmento', 'fecha_carga']
Carga completada correctamente
